# EDA — 전국체육시설 안전점검 정보 (TODZ_API_FACI_SAFETY)

한국외국어대학교 26-2학기 「스포츠융합산업프로젝트실습」 캡스톤
**전국 체육시설 안전점검 데이터를 활용한 시설 안전위험 예측 및 점검자원 우선순위 최적화**

이 노트북은 `src/eda.py`의 함수를 그대로 불러와 실행한다(단일 소스 유지 — 로직 수정은
`src/eda.py`에서). 데이터 정의는 `docs/data_dictionary.md`, 그림/발견 요약은
`docs/eda_findings.md` 참고.

In [ ]:
import sys
sys.path.append("../src")
import eda
import pandas as pd
pd.set_option("display.max_columns", 50)

df_raw = eda.load_raw()
df = eda.clean(df_raw)
print("행/열:", df.shape)
df.head(3)

## 1. 데이터 개요
결측률, 운영상태, 시설구분, 자율점검대상여부를 한눈에 본다.

In [ ]:
fig_path = eda.fig_overview(df)
from IPython.display import Image
Image(filename=str(fig_path))

## 2. 타겟 변수(`schk_tot_grd_nm`) 재정의

- 원본은 3-클래스(양호/주의/사용중지)지만 `사용중지`가 전국 23건뿐이라 3-클래스 분류는
  통계적으로 무의미함에 가깝다.
- **이진 타겟(`risk_binary`)으로 재정의**: 양호=0, (주의+사용중지)=1.
- 점검 이력이 없는 33.6%는 결측으로 별도 처리(라벨 없음).

In [ ]:
fig_path = eda.fig_target(df)
Image(filename=str(fig_path))

In [ ]:
print("이진 타겟 분포 (결측 제외):")
print(df["risk_binary"].value_counts(dropna=True))
print()
print("전체 위험 비율: {:.2f}%".format(df["risk_binary"].mean() * 100))

## 3. 범주형 예측 피처 분포
업종, 시설유형, 시도, 시군구, 운영형태, 실내외구분.

In [ ]:
fig_path = eda.fig_categorical(df)
Image(filename=str(fig_path))

## 4. 범주별 위험군 비율 — 핵심 인사이트

단순 빈도가 아니라 **"이 범주에 속한 시설 중 몇 %가 위험군인가"**를 봐야 실제 예측에 쓸모가
있다. 최소 표본 30건 이상인 범주만 포함(사군소표본 노이즈 제거).

In [ ]:
fig_path = eda.fig_risk_rate(df)
Image(filename=str(fig_path))

In [ ]:
print("업종별 위험 비율 Top 10 (n>=30):")
display(eda._risk_rate_by(df, "fcob_nm", min_n=30, topn=10))
print()
print("시도별 위험 비율:")
display(eda._risk_rate_by(df, "cp_nm", min_n=30, topn=16))

## 5. 수치형(파생) 피처 분포
면적(정제), 시설연령, 마지막 점검 후 경과일수.

In [ ]:
fig_path = eda.fig_numeric(df)
Image(filename=str(fig_path))

## 6. 위험군 여부에 따른 수치형 피처 비교

In [ ]:
fig_path = eda.fig_numeric_vs_target(df)
Image(filename=str(fig_path))

In [ ]:
summary = df.groupby("risk_label")[["area_sqm", "facility_age_years", "days_since_inspection"]].median()
summary

## 7. 결측치 패턴
주요 컬럼 간 결측이 함께 발생하는 블록 구조가 있는지 확인.

In [ ]:
fig_path = eda.fig_missing_pattern(df)
Image(filename=str(fig_path))

## 8. 수치형 피처 상관관계

In [ ]:
fig_path = eda.fig_correlation(df)
Image(filename=str(fig_path))

## 결론 요약

1. **타겟**: `schk_tot_grd_nm` → 이진 위험군(`risk_binary`)으로 재정의. 위험 비율 3.85%로
   심각한 클래스 불균형 — `class_weight`/리샘플링 필요.
2. **분석 범위**: 폐업/휴업/운영폐쇄 시설(27.1%)은 자원배분 최적화 관점에서 의미가 없어
   `faci_stat_nm == 정상운영`으로 필터링 권장.
3. **라벨 결측(33.6%, 정상운영만도 23.3%)**: 학습에서 제외하되, "왜 점검을 안 받았는가"
   자체도 추후 별도 가설로 남겨둠.
4. **신호가 있는 피처**: 업종(빙상장/사격장/육상경기장/수영장 등 위험비율 20%+),
   시도(세종 10.6%로 최고), 시설구분(등록업 24.2% > 공공 10.5% > 신고업 2.5%),
   시설연령(위험군이 더 낮은 연령대에 몰림 — 반직관적, 추가 조사 필요).
5. 상세 수치·해석은 `docs/eda_findings.md`, 다음 단계는 `docs/WORKFLOW.md` 참고.